# Imports

In [1]:
from dotenv import load_dotenv
import os

import pandas as pd
import numpy as np
from pandas_datareader import data as pdr

import plotly.graph_objs as go
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from scipy import stats

from util_functions import find_cols, remove_outliers, get_outliers
from data_functions import  get_fred_data

# Env

In [2]:
load_dotenv()

True

In [3]:
wb_commodities_data_file_url = os.getenv('WB_COMMODITIES_DATA_FILE_URL')

In [4]:
wb_commodities_data_file_url

'https://thedocs.worldbank.org/en/doc/5d903e848db1d1b83e0ec8f744e55570-0350012021/related/CMO-Historical-Data-Monthly.xlsx'

In [5]:
wb_commodities_data_local_filepath = os.getenv('WB_COMMODITIES_DATA_LOCAL_FILEPATH')

In [6]:
wb_commodities_data_local_filepath

'/Users/zayankhan/Desktop/1212 Analytics/Cotton Data/World Bank/CMO-Historical-Data-Monthly.xlsx'

# Load

In [7]:
wb_commodities_df = pd.read_excel(wb_commodities_data_local_filepath, sheet_name=1, skiprows=4)

In [8]:
wb_commodities_df

,Unnamed: 0,"Crude oil, average","Crude oil, Brent","Crude oil, Dubai","Crude oil, WTI","Coal, Australian","Coal, South African **","Natural gas, US","Natural gas, Europe","Liquefied natural gas, Japan",...,Aluminum,"Iron ore, cfr spot",Copper,Lead,Tin,Nickel,Zinc,Gold,Platinum,Silver
0,NaN,($/bbl),($/bbl),($/bbl),($/bbl),($/mt),($/mt),($/mmbtu),($/mmbtu),($/mmbtu),...,($/mt),($/dmtu),($/mt),($/mt),($/mt),($/mt),($/mt),($/troy oz),($/troy oz),($/troy oz)
1,1960M01,1.63,1.63,1.63,…,…,…,0.14,0.404774,…,...,511.471832,11.42,715.4,206.1,2180.4,1631,260.8,35.27,83.5,0.9137
2,1960M02,1.63,1.63,1.63,…,…,…,0.14,0.404774,…,...,511.471832,11.42,728.19,203.7,2180.4,1631,244.9,35.27,83.5,0.9137
3,1960M03,1.63,1.63,1.63,…,…,…,0.14,0.404774,…,...,511.471832,11.42,684.94,210.3,2173.8,1631,248.7,35.27,83.5,0.9137
4,1960M04,1.63,1.63,1.63,…,…,…,0.14,0.404774,…,...,511.471832,11.42,723.11,213.6,2178.2,1631,254.6,35.27,83.5,0.9137
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
767,2023M11,81.354333,83.183,83.45,77.43,126.82,109,2.7084,14.485055,12.723407,...,2202.26,131.07,8189.59,2188.46,24167.86,17027.36,2543.61,1984.11,905.75,23.488
768,2023M12,75.719333,77.858,77.22,72.08,141.82,108.8294,2.5265,11.506658,14.439545,...,2182.43,137.05,8399.94,2026.91,24600.26,16460.84,2502.39,2026.18,935.47,23.878
769,2024M01,77.672333,80.227,78.86,73.93,124.9,106.75,3.1806,9.559614,14.344154,...,2192.82,135.82,8338.88,2086.12,25099.84,16103.83,2515.42,2034.04,925.86,22.916
770,2024M02,80.548,83.764,81.18,76.7,124.22,105.193,1.7211,8.148381,13.644993,...,2179.46,124.39,8304.95,2079.83,26104.1,16338.46,2360.09,2023.24,894.29,22.657


# Transform

### Update col names to include metrics and to numeric

In [9]:
new_header = wb_commodities_df.columns + wb_commodities_df.iloc[0]

In [10]:
wb_commodities_df.columns = new_header

In [11]:
wb_commodities_df = wb_commodities_df.drop(0)

In [12]:
wb_commodities_df

,NaN,"Crude oil, average($/bbl)","Crude oil, Brent($/bbl)","Crude oil, Dubai($/bbl)","Crude oil, WTI($/bbl)","Coal, Australian($/mt)","Coal, South African **($/mt)","Natural gas, US($/mmbtu)","Natural gas, Europe($/mmbtu)","Liquefied natural gas, Japan($/mmbtu)",...,Aluminum($/mt),"Iron ore, cfr spot($/dmtu)",Copper($/mt),Lead($/mt),Tin($/mt),Nickel($/mt),Zinc($/mt),Gold($/troy oz),Platinum($/troy oz),Silver($/troy oz)
1,1960M01,1.63,1.63,1.63,…,…,…,0.14,0.404774,…,...,511.471832,11.42,715.4,206.1,2180.4,1631,260.8,35.27,83.5,0.9137
2,1960M02,1.63,1.63,1.63,…,…,…,0.14,0.404774,…,...,511.471832,11.42,728.19,203.7,2180.4,1631,244.9,35.27,83.5,0.9137
3,1960M03,1.63,1.63,1.63,…,…,…,0.14,0.404774,…,...,511.471832,11.42,684.94,210.3,2173.8,1631,248.7,35.27,83.5,0.9137
4,1960M04,1.63,1.63,1.63,…,…,…,0.14,0.404774,…,...,511.471832,11.42,723.11,213.6,2178.2,1631,254.6,35.27,83.5,0.9137
5,1960M05,1.63,1.63,1.63,…,…,…,0.14,0.404774,…,...,511.471832,11.42,684.75,213.4,2162.7,1631,253.8,35.27,83.5,0.9137
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
767,2023M11,81.354333,83.183,83.45,77.43,126.82,109,2.7084,14.485055,12.723407,...,2202.26,131.07,8189.59,2188.46,24167.86,17027.36,2543.61,1984.11,905.75,23.488
768,2023M12,75.719333,77.858,77.22,72.08,141.82,108.8294,2.5265,11.506658,14.439545,...,2182.43,137.05,8399.94,2026.91,24600.26,16460.84,2502.39,2026.18,935.47,23.878
769,2024M01,77.672333,80.227,78.86,73.93,124.9,106.75,3.1806,9.559614,14.344154,...,2192.82,135.82,8338.88,2086.12,25099.84,16103.83,2515.42,2034.04,925.86,22.916
770,2024M02,80.548,83.764,81.18,76.7,124.22,105.193,1.7211,8.148381,13.644993,...,2179.46,124.39,8304.95,2079.83,26104.1,16338.46,2360.09,2023.24,894.29,22.657


### Set YearMonth index as datetime index

In [13]:
wb_commodities_df = wb_commodities_df.rename(columns={wb_commodities_df.columns[0]: 'YearMonth'})

In [14]:
# Convert 'date' column to datetime format
wb_commodities_df['YearMonth'] = pd.to_datetime(wb_commodities_df['YearMonth'], format='%YM%m')

In [15]:
wb_commodities_df.set_index(wb_commodities_df['YearMonth'], inplace=True)

In [16]:
wb_commodities_df.drop('YearMonth', axis=1, inplace=True)

In [17]:
wb_commodities_df = wb_commodities_df.apply(pd.to_numeric, errors='coerce')

### Find cols

In [18]:
find_cols(wb_commodities_df, 'Cotton')

['Cotton, A Index($/kg)']

# Corellations

In [19]:
px.box(wb_commodities_df[['Cotton, A Index($/kg)']].loc['2000-01-01':])

In [20]:
scaler = StandardScaler()

In [21]:
wb_commodities_outliers_rm_df = remove_outliers(wb_commodities_df)

In [22]:
wb_commodities_outliers_df = get_outliers(wb_commodities_df)

In [23]:
px.box(wb_commodities_outliers_rm_df[['Cotton, A Index($/kg)']])

In [24]:
# Scale the data
wb_commodities_outliers_rm_data_scaled = scaler.fit_transform(wb_commodities_outliers_rm_df)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning:

invalid value encountered in divide

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning:

invalid value encountered in divide

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning:

invalid value encountered in divide



In [25]:
# Convert scaled_data back to DataFrame
wb_commodities_outliers_rm_df_scaled = pd.DataFrame(wb_commodities_outliers_rm_data_scaled, columns=wb_commodities_outliers_rm_df.columns, index=wb_commodities_outliers_rm_df.index)

In [26]:
# Calculate correlations
wb_commodities_outliers_rm_correlation_matrix = wb_commodities_outliers_rm_df_scaled.corr()

In [27]:
cotton_correlations = wb_commodities_outliers_rm_correlation_matrix['Cotton, A Index($/kg)'].sort_values(ascending=False)

In [28]:
cotton_top_correlations = list(cotton_correlations[:5].keys())
cotton_top_correlations

['Cotton, A Index($/kg)',
 'Coffee, Arabica($/kg)',
 'Soybeans($/mt)',
 'Sugar, US($/kg)',
 'Maize($/mt)']

In [29]:
# Create traces for each commodity
traces = []
for commodity in cotton_top_correlations:
    trace = go.Scatter(x=wb_commodities_outliers_rm_df_scaled.loc['2000-01-01':].index, y=wb_commodities_outliers_rm_df_scaled.loc['2000-01-01':][commodity], mode='lines', name=commodity)
    traces.append(trace)

# Define layout for the plot
layout = go.Layout(
    title='Cotton Top Correlations',
    xaxis=dict(title='YearMonth'),
    yaxis=dict(title='Scaled Spot Px')
)

# Create figure and plot
fig = go.Figure(data=traces, layout=layout)

# Display the interactive plot
fig.show()


In [30]:
wb_commodities_df[['Cotton, A Index($/kg)']]

,"Cotton, A Index($/kg)"
YearMonth,
1960-01-01,0.648600
1960-02-01,0.645300
1960-03-01,0.646400
1960-04-01,0.643500
1960-05-01,0.646800
...,...
2023-11-01,1.994299
2023-12-01,1.995402
2024-01-01,2.030014


# Cotton DF

In [31]:
wb_cotton_df = wb_commodities_outliers_rm_df[['Cotton, A Index($/kg)']].loc['2000-01-01':].ffill()
#fill with dynamic values - replacing outliers with winsorised estimators 

In [32]:
cotton_trace = go.Scatter(x=wb_cotton_df.index, y=wb_cotton_df['Cotton, A Index($/kg)'], mode='lines', name='Cotton, A Index($/kg)')
# Define layout for the plot
layout = go.Layout(
    title='Cotton, A Index($/kg)',
    xaxis=dict(title='YearMonth'),
    yaxis=dict(title='$ US')
)

# Create figure and plot
fig = go.Figure(data=cotton_trace, layout=layout)

# Display the interactive plot
fig.show()

## Adjust for PPI (Inflation) - Cotton factors

### Create Cotton PPI metric 

In [52]:
ppi_codes_dict = {'finished_cotton': 'WPU034201', 'spun_cotton_yarns':'WPU032601', 'carded_spun_yarns': 'WPU03260113'}

In [53]:
ppi_values_cotton = get_fred_data(wb_cotton_df.index[0], wb_cotton_df.index[-1], ppi_codes_dict)
ppi_values_cotton['spun_cotton_yarns'] = ppi_values_cotton['spun_cotton_yarns'].ffill().bfill()

In [54]:
ppi_values_cotton

,finished_cotton,spun_cotton_yarns,carded_spun_yarns
DATE,,,
2000-01-01,114.700,92.300,NaN
2000-02-01,114.200,92.200,NaN
2000-03-01,113.100,92.500,NaN
2000-04-01,113.700,92.400,NaN
2000-05-01,113.000,92.400,NaN
...,...,...,...
2023-11-01,195.486,152.708,NaN
2023-12-01,195.613,152.708,NaN
2024-01-01,196.064,152.708,NaN


In [36]:
ppi_values_cotton['Composite PPI'] = ppi_values_cotton.mean(axis=1)
#ToDo: weighted average with 50% weighting to spun yarn ppi

In [37]:
ppi_values_cotton

,finished_cotton,spun_cotton_yarns,carded_spun_yarns,Composite PPI
DATE,,,,
2000-01-01,114.700,92.300,NaN,103.5000
2000-02-01,114.200,92.200,NaN,103.2000
2000-03-01,113.100,92.500,NaN,102.8000
2000-04-01,113.700,92.400,NaN,103.0500
2000-05-01,113.000,92.400,NaN,102.7000
...,...,...,...,...
2023-11-01,195.486,152.708,NaN,174.0970
2023-12-01,195.613,152.708,NaN,174.1605
2024-01-01,196.064,152.708,NaN,174.3860


In [38]:
ppi_values_cotton.index.names = ['YearMonth']

In [39]:
wb_cotton_df = wb_cotton_df.merge(ppi_values_cotton, left_index=True, right_index=True)

In [40]:
wb_cotton_df

,"Cotton, A Index($/kg)",finished_cotton,spun_cotton_yarns,carded_spun_yarns,Composite PPI
YearMonth,,,,,
2000-01-01,1.045700,114.700,92.300,NaN,103.5000
2000-02-01,1.184500,114.200,92.200,NaN,103.2000
2000-03-01,1.263200,113.100,92.500,NaN,102.8000
2000-04-01,1.293900,113.700,92.400,NaN,103.0500
2000-05-01,1.333600,113.000,92.400,NaN,102.7000
...,...,...,...,...,...
2023-11-01,1.994299,195.486,152.708,NaN,174.0970
2023-12-01,1.995402,195.613,152.708,NaN,174.1605
2024-01-01,2.030014,196.064,152.708,NaN,174.3860


In [41]:
base_ppi_cotton = wb_cotton_df['Composite PPI'].iloc[-1]

In [55]:
#so you do it with base ppi (todays price) vs a past date is bc the ppi index is a relational index so from x value it shows the % value the px on one date would be more or less
wb_cotton_df['Cotton, A Index($/kg) Inflation Adj'] = wb_cotton_df['Cotton, A Index($/kg)'] * (base_ppi_cotton / wb_cotton_df['Composite PPI'])

In [43]:
wb_cotton_df

,"Cotton, A Index($/kg)",finished_cotton,spun_cotton_yarns,carded_spun_yarns,Composite PPI,"Cotton, A Index($/kg) Inflation Adj"
YearMonth,,,,,,
2000-01-01,1.045700,114.700,92.300,NaN,103.5000,1.777645
2000-02-01,1.184500,114.200,92.200,NaN,103.2000,2.019452
2000-03-01,1.263200,113.100,92.500,NaN,102.8000,2.162007
2000-04-01,1.293900,113.700,92.400,NaN,103.0500,2.209179
2000-05-01,1.333600,113.000,92.400,NaN,102.7000,2.284722
...,...,...,...,...,...,...
2023-11-01,1.994299,195.486,152.708,NaN,174.0970,2.015474
2023-12-01,1.995402,195.613,152.708,NaN,174.1605,2.015853
2024-01-01,2.030014,196.064,152.708,NaN,174.3860,2.048168


In [44]:
# Create traces
cotton_px_trace= go.Scatter(
    x = wb_cotton_df.index,
    y = wb_cotton_df['Cotton, A Index($/kg)'],
    mode = 'lines',
    name = 'Cotton, A Index($/kg)'
)

cotton_adj_px_trace = go.Scatter(
    x = wb_cotton_df.index,
    y = wb_cotton_df['Cotton, A Index($/kg) Inflation Adj'],
    mode = 'lines',
    name = 'Cotton, A Index($/kg) Inflation Adj'
)

In [45]:
# Create the layout
layout = go.Layout(
    title = 'Cotton, A Index vs Cotton, A Index: Inflation Adj ($/kg)',
    xaxis = dict(title = 'Year-Month'),
    yaxis = dict(title = '($/kg)')
)

In [46]:
# Create the figure
cotton_fig = go.Figure(data = [cotton_px_trace, cotton_adj_px_trace], layout = layout)

# Plot the figure
cotton_fig.show()